In [1]:
import pandas as pd
import numpy as np
import optuna
import shap
import xgboost as xgb
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, average_precision_score, make_scorer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import GroupKFold, cross_validate
import miceforest as mf
import warnings, os, datetime

In [ ]:
FEATURES_TO_SELECT = ['Age', 'Sex', 'Рост', 'Вес', 'ЧСС (b)', 'Систолическое АД(b)', 'Инфаркт миокарда в анамнезе (<3)', 'Инфаркт миокарда в анамнезе (>3)', 'Инфаркт миокарда со стентированием в анамнезе', 'ОНМК (иш) в анамнезе',]
MICE_COLUMNS = ['Age', 'Sex', 'Рост', 'Вес', 'ЧСС (b)', 'Систолическое АД(b)', 'Инфаркт миокарда в анамнезе (<3)', 'Инфаркт миокарда в анамнезе (>3)', 'Инфаркт миокарда со стентированием в анамнезе', 'ОНМК (иш) в анамнезе',]
TARGET = 'КШ развился в реанимации'
N_FOLDS = 5
OPTUNA_TRIALS = 300
RANDOM_STATE = 42

In [24]:
dataAllFull = pd.read_excel('data_raw/side_project/DataSet_V49_9_03_26.xlsx')
data = dataAllFull.loc[
    (dataAllFull['STEMI (Новый)'] == 'Да') &
    (dataAllFull['Наличие в файле'] == 'Да') &
    (dataAllFull['ЧКВ'] == 'Да')
].copy()

print(f'Filtered v49: {data.shape}')

killip_df = pd.read_excel('data_raw/side_project/filtered_killip.xlsx')
killip_df = killip_df.drop(columns='Unnamed: 0', errors='ignore')

key = 'Код пациента'
common_cols = set(data.columns) & set(killip_df.columns)
killip_sub = killip_df[[key] + [c for c in killip_df.columns if c not in common_cols]].copy()
final_df = data.merge(killip_sub, on=key, how='inner')
print(f'Merged: {final_df.shape}')
print(f'Target: {final_df[TARGET].value_counts().to_dict()}')

Filtered v49: (7524, 473)
Merged: (5588, 510)
Target: {0: 5406, 1: 182}


In [25]:
to_cat = final_df.select_dtypes(include=['object']).columns
final_df[to_cat] = final_df[to_cat].astype('category')

final_df.drop(['Диагноз_ИМ_Дата', 'Дата_STEMI', 'Дата_направления_Общий_анализ_крови', 'Дата_взятия_биоматериала_Общий_анализ_крови', 'Дата_выполнения_Общий_анализ_крови', 'Дата_направления_Общий_анализ_крови_экспрес', 'Дата_взятия_биоматериала_Общий_анализ_крови_экспрес', 'Дата_выполнения_Общий_анализ_крови_экспрес', 'Начало_операции_ИБ_Новый', 'Конец_операции_ИБ_Новый', 'Продолжительность_операции', 'дата_поступления', 'дата_выписки', 'длительность_нахождения_в_стационаре', 'дата_смерти', 'Поступление_в_реанимацию', 'Выписка_из_реанимации', 'Количество_дней_в_реанимации', 'Дата_и_время_развития_SOFA_8_и_более', 'Время_введения_первого_антибиотика'], inplace=True, errors='ignore')

final_df = final_df.loc[:, final_df.isnull().mean() < 0.8]

In [33]:
import pandas as pd
import miceforest as mf
import re

# Функция для очистки имени колонки: оставляет только буквы (латиница/кириллица), цифры и подчеркивания
def clean_column_name(name):
    # Заменяем все символы, кроме букв, цифр и подчеркивания, на '_'
    cleaned = re.sub(r'[^a-zA-Zа-яА-Я0-9_><]', '_', name)
    # Убираем повторяющиеся подчёркивания
    cleaned = re.sub(r'_+', '_', cleaned)
    # Убираем подчёркивания в начале и конце
    cleaned = cleaned.strip('_')
    return cleaned

# Сохраняем оригинальные имена
original_cols = final_df.columns.tolist()

# Создаём словарь для переименования: старые имена -> новые безопасные
rename_dict = {col: clean_column_name(col) for col in original_cols}
# Проверяем уникальность новых имён (на случай, если два разных имени дадут одинаковое после очистки)
# Если коллизия, можно добавить суффикс
new_names = list(rename_dict.values())
if len(set(new_names)) < len(new_names):
    # Добавляем суффиксы для повторяющихся
    from collections import Counter
    counts = Counter(new_names)
    for col, new_name in rename_dict.items():
        if counts[new_name] > 1:
            # Найдём порядковый номер для этого имени
            idx = [n for n in new_names if n == new_name].index(new_name) + 1
            rename_dict[col] = f"{new_name}_{idx}"

# Переименовываем колонки в датафрейме
final_df = final_df.replace([np.inf, -np.inf], np.nan)
bad_cols = [c for c in final_df.columns if final_df[c].nunique() <= 1]
final_df = final_df.drop(columns=bad_cols)

df_renamed = final_df.rename(columns=rename_dict)
df_renamed = df_renamed.select_dtypes(exclude=['datetime64', 'datetimetz'])


# Теперь работаем с переименованным датафреймом
target_cols_orig = FEATURES_TO_SELECT

# Преобразуем оригинальные имена в новые
target_cols = [rename_dict[col] for col in target_cols_orig]

predictor_cols = [col for col in df_renamed.columns if col not in target_cols]
# Исключаем объектные колонки, если они есть
predictor_cols = [
    col for col in predictor_cols 
    if df_renamed[col].dtypes.name not in ['object', 'datetime64', 'timedelta64']
]

variable_schema = {col: predictor_cols for col in target_cols}

kernel = mf.ImputationKernel(
    data=df_renamed,
    variable_schema=variable_schema,
    random_state=42,
    model={'model': 'xgboost'}
)

kernel.mice(iterations=2, verbose=True)
df_imputed_renamed = kernel.complete_data(dataset=0)

# Возвращаем исходные имена колонок
# Для этого создаём обратный словарь
reverse_rename = {v: k for k, v in rename_dict.items()}
df_imputed = df_imputed_renamed.rename(columns=reverse_rename)

TypeError: ImputationKernel.__init__() got an unexpected keyword argument 'model'

In [28]:
df_imputed

NameError: name 'df_imputed' is not defined